# Archiver

The Archiver class is responsible to insert the model predictions into an xarray dataset at the correct coordinates and save the result to disk.
To do this, we use the `crest.archiver.Archiver` class. By passing in the model predictions from the predict() method of the `crest.model.Model` class, we can start writing the predictions to disk.

Here, we are going to demonstrate that how the Archiver works. But, before going into that part, let's first create two dummy Datafiles to insert the predictions into:

In [1]:
 # Ensure we can import crest
%cd -q ..
import numpy as np
import pandas as pd
import xarray as xr
import dask.array as da
import tensorflow as tf

from crest.data.loading.Datafile import Datafile
from crest.data.loading.Dataset import Dataset
from crest.model.HierarchalTensorGraph import HierarchalTensorGraph as HTG
from crest.model.Model import Model
from crest.archiver.Archiver import Archiver

In [2]:
latitude = np.linspace(-90, 90, 180)
longitude = np.linspace(-180, 180, 180)
datetime = pd.date_range('2015-04-01 00:00:00',
                          '2015-08-31 23:00:00',
                          freq='H').values
                   

var1 = da.full(shape=(datetime.size, latitude.size, longitude.size), 
               fill_value=np.nan
               )

ds = xr.Dataset(
                data_vars=dict(
                    var1=(["datetime", "latitude", "longitude"], var1)
                    ),
                coords=dict(
                    datetime=datetime,
                    latitude=(["latitude"], latitude),
                    longitude=(["longitude"], longitude)
                    )
                ).chunk('auto')

df = Datafile(location=ds,
              extent={'latitude': [24.396, 49.384], 
                      'longitude': [-124.848, -66.885],
                      'datetime':[np.datetime64('2015-04-01 00:00:00'), np.datetime64('2015-05-01 23:00:01')]
              },
              invalid_value = np.nan,
              features = ['var1']
             )


latitude = np.linspace(-90, 90, 360)
longitude = np.linspace(-180, 180, 200)
datetime = pd.date_range('2015-04-01 00:00:00',
                          '2015-08-31 23:00:00',
                          freq='H').values
                   

var2 = da.full(shape=(datetime.size, latitude.size, longitude.size), 
               fill_value=np.nan
               )

ds = xr.Dataset(
                data_vars=dict(
                    var2=(["datetime", "latitude", "longitude"], var2)
                    ),
                coords=dict(
                    datetime=datetime,
                    latitude=(["latitude"], latitude),
                    longitude=(["longitude"], longitude)
                    )
                ).chunk('auto')

df1 = Datafile(location=ds,
              extent={'latitude': [24.396, 49.384], 
                      'longitude': [-124.848, -66.885],
                      'datetime':[np.datetime64('2015-04-01 00:00:00'), np.datetime64('2015-05-01 23:00:01')]
              },
              invalid_value = np.nan,
              features = ['var2']
             )

Now, we create a model and get its prediction by a dummy input of shape (batch, datetime, latitude, longitude, channels) = (5,10,1,1,5):

In [3]:
class layer1:
    def __init__(self):
        self.name='layer1'
    
    def __call__(self, 
             inputs):
        x = tf.keras.layers.Lambda(lambda d: tf.squeeze(d))(inputs['model_input'])
        x = {'model_output': tf.keras.layers.LSTM(1)(x)}
        
        return x

graph = HTG(
            name='dummy_model',
            inputs={'model_input': tf.TensorSpec(shape=[5,10, 1, 1, 5])},
            outputs={'model_output' : tf.TensorSpec(shape=[5,1])}
            )


l1 = layer1()
graph.add_edge('input', l1)
graph.add_edge(l1.name, 'output')
model = Model(graph)

build_kwargs= {'optimizer' : 'Adam', 
               'loss'      : 'mse'}
model.build(**build_kwargs)

predictions = model.predict_on_batch({'model_input': np.random.random((5,10,1,1,5))})

2023-10-30 10:24:51.138768: W tensorflow/core/platform/profile_utils/cpu_utils.cc:128] Failed to get CPU frequency: 0 Hz


Now we add the coordinates info to the predictions dictionary. Usually, the coordinates info and the schemer dataset are from a same source of data. But, here we get the coordinates info from different datafile than the schemer to show that the Archiver is able to manage the discrepancy. The coordinates info is from the first datafile and we left the datafile_index empty so the datafile with finest resolution (second datafile) will be chosen as the schemer:

In [4]:
predictions['latitude'] = np.random.choice(df.data['latitude'], (5,1))
predictions['longitude'] = np.random.choice(df.data['longitude'], (5,1))
predictions['datetime'] = np.random.choice(df.data['datetime'], (5,1))

Next, we will create an instance from the Archiver class and write the predictions into the dataset and save it using archive() method:

In [ ]:
# We define a dataset of the two datafiles as the  
# data schemer which is used to specify the reference data 
# framework to insert the predictions into that. 
# If the argument is ignored by the user, the Archiver 
# tries to use the dataset at the output_path as the schemer 
# if exists.

a = Archiver(output_path='output.zarr',
             data_schemer=Dataset([df, df1]))

# Using the archive() method we write the predictions
# (This method should be called in a batcher loop)
a.archive(predictions=predictions)

Here, we verify the Archiver.archive() results:

In [ ]:
output_data = xr.open_zarr('output.zarr')['model_output']
written_value = output_data.sel({'datetime': predictions['datetime'][1],
                                 'latitude': predictions['latitude'][1],
                                 'longitude': predictions['longitude'][1]},
                                method='nearest'
                                ).to_numpy()
original_value = predictions['model_output'][1]

print(f'original_value vs. written_value: {original_value} vs. {written_value}')